Step 1: Load Dataset

In [3]:
import pandas as pd

file_path = r"C:\Users\saksh\OneDrive\Desktop\House_Price_Prediction\data\housing_data.csv"

df = pd.read_csv(file_path)

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
2,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
3,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
4,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished


Step 2: Basic Info

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 547 entries, 0 to 546
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             547 non-null    int64
 1   area              547 non-null    int64
 2   bedrooms          547 non-null    int64
 3   bathrooms         547 non-null    int64
 4   stories           547 non-null    int64
 5   mainroad          547 non-null    str  
 6   guestroom         547 non-null    str  
 7   basement          547 non-null    str  
 8   hotwaterheating   547 non-null    str  
 9   airconditioning   547 non-null    str  
 10  parking           547 non-null    int64
 11  prefarea          547 non-null    str  
 12  furnishingstatus  547 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.7 KB


Step 3: Check Missing Values

In [3]:
df.isnull().sum()

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Step 4: Basic Cleaning

In [4]:
df = df.drop_duplicates()
df.shape

(545, 13)

Step 5: Convert YES/NO → 1/0

In [5]:
binary_cols = ['mainroad', 'guestroom', 'basement', 
               'hotwaterheating', 'airconditioning', 'prefarea']

for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,furnished
2,12250000,8960,4,4,4,1,0,0,0,1,3,0,furnished
3,12250000,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
4,12215000,7500,4,2,2,1,0,1,0,1,3,1,furnished
5,11410000,7420,4,1,2,1,1,1,0,1,2,0,furnished


Step 6: Handle Categorical Column (furnishingstatus)

In [6]:
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,False,False
2,12250000,8960,4,4,4,1,0,0,0,1,3,0,False,False
3,12250000,9960,3,2,2,1,0,1,0,0,2,1,True,False
4,12215000,7500,4,2,2,1,0,1,0,1,3,1,False,False
5,11410000,7420,4,1,2,1,1,1,0,1,2,0,False,False


Step 7: Train-Test Split

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
import os
print(os.listdir())

['house_price_prediction.ipynb', 'mlflow.db', 'mlruns']


In [12]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("House_Price_Prediction")

<Experiment: artifact_location='file:c:/Users/saksh/OneDrive/Desktop/House_Price_Prediction/notebooks/mlruns/833833366324037801', creation_time=1774007876557, experiment_id='833833366324037801', last_update_time=1774007876557, lifecycle_stage='active', name='House_Price_Prediction', tags={}, workspace='default'>

Step 8: Train Simple Model (Linear Regression)

In [13]:
import mlflow.sklearn

with mlflow.start_run(run_name="Linear_Regression"):

    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error, r2_score

    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)

    y_pred_lr = model_lr.predict(X_test)

    mae_lr = mean_absolute_error(y_test, y_pred_lr)
    r2_lr = r2_score(y_test, y_pred_lr)

    mlflow.log_metric("MAE", mae_lr)
    mlflow.log_metric("R2", r2_lr)

    mlflow.sklearn.log_model(model_lr, "LinearRegression_model")

    print("Linear Regression -> MAE:", mae_lr, "R2:", r2_lr)

2026/03/20 17:28:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 17:28:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression -> MAE: 970043.4039201637 R2: 0.6529242642153186


Step 8B: Ridge Regression

In [14]:
from sklearn.linear_model import Ridge

with mlflow.start_run(run_name="Ridge_Regression"):

    model_ridge = Ridge(alpha=1.0)
    model_ridge.fit(X_train, y_train)


    y_pred_ridge = model_ridge.predict(X_test)

    mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
    r2_ridge = r2_score(y_test, y_pred_ridge)

    mlflow.log_metric("MAE", mae_ridge)
    mlflow.log_metric("R2", r2_ridge)

    mlflow.sklearn.log_model(model_ridge, "RidgeRegression_model")

    print("Ridge -> MAE:", mae_ridge, "R2:", r2_ridge)

2026/03/20 17:29:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 17:29:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ridge -> MAE: 970245.6821765776 R2: 0.6524978002155009


Step 8C: Decision Tree

In [15]:
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name="Decision_Tree"):

    model_dt = DecisionTreeRegressor(random_state=42)
    model_dt.fit(X_train, y_train)

    y_pred_dt = model_dt.predict(X_test)

    mae_dt = mean_absolute_error(y_test, y_pred_dt)
    r2_dt = r2_score(y_test, y_pred_dt)

    mlflow.log_metric("MAE", mae_dt)
    mlflow.log_metric("R2", r2_dt)

    mlflow.sklearn.log_model(model_dt, "DecisionTree_model")

    print("Decision Tree -> MAE:", mae_dt, "R2:", r2_dt)

2026/03/20 17:29:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 17:29:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree -> MAE: 1195266.0550458715 R2: 0.4771459275854347


Step 9: Compare Models

In [16]:
results = {
    "Linear": (mae_lr, r2_lr),
    "Ridge": (mae_ridge, r2_ridge),
    "Decision Tree": (mae_dt, r2_dt)
}

for model, (mae, r2) in results.items():
    print(f"{model} -> MAE: {mae}, R2: {r2}")

Linear -> MAE: 970043.4039201637, R2: 0.6529242642153186
Ridge -> MAE: 970245.6821765776, R2: 0.6524978002155009
Decision Tree -> MAE: 1195266.0550458715, R2: 0.4771459275854347


In [17]:
# Find best model based on R2
best_model_name = max(results, key=lambda x: results[x][1])

print("Best Model:", best_model_name)

Best Model: Linear


In [18]:
# Save best model in MLflow

best_model_map = {
    "Linear": model_lr,
    "Ridge": model_ridge,
    "Decision Tree": model_dt
}

best_model = best_model_map[best_model_name]
best_mae, best_r2 = results[best_model_name]

with mlflow.start_run(run_name="Best_Model"):

    mlflow.log_param("best_model", best_model_name)
    mlflow.log_metric("best_MAE", best_mae)
    mlflow.log_metric("best_R2", best_r2)

    mlflow.sklearn.log_model(best_model, "best_model")

print("✅ Best model logged")

2026/03/20 17:33:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 17:33:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Best model logged


In [19]:
import joblib

# Save best model
joblib.dump(best_model, "best_model.pkl")

['best_model.pkl']